# Banking Intent — Fine-tune Qwen3-8B on Google Colab

**Before running:**
1. Runtime → Change runtime type → **A100** (or L4)
2. Colab → left sidebar → 🔑 Secrets: add `HF_TOKEN` (HuggingFace token with write access)
3. Fill in `GITHUB_REPO_URL` and `HF_REPO_ID` in the next cell

In [ ]:
# === CONFIG — change these two lines ===
GITHUB_REPO_URL = "https://github.com/nguyenvmthien/banking-intent-unsloth.git"
HF_REPO_ID      = "minhthien/banking-intent-unsloth"
# ========================================

import os
WORKING_DIR = f"/content/{GITHUB_REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')}"
print(f"Repo will be cloned to: {WORKING_DIR}")

## 1. Environment setup

In [ ]:
import subprocess, sys

def run(cmd, capture=True):
    result = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if result.returncode != 0:
        print(result.stderr[-3000:] if capture else "")
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout if capture else ""

run('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
run('pip install -q trl peft datasets langsmith huggingface_hub pandas scikit-learn tqdm pyyaml')
print("Dependencies installed.")

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN loaded.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Checkpoints are saved here — survives session restarts
CHECKPOINT_DIR = "/content/drive/MyDrive/banking-intent-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

## 2. Clone repo and prepare data

In [ ]:
run(f'git clone {GITHUB_REPO_URL} {WORKING_DIR}')
os.chdir(f"{WORKING_DIR}/scripts")
print(f"Repo cloned to {WORKING_DIR}")

In [ ]:
run('python preprocess_data.py')
print("Data ready.")

## 3. Configure HF repo for checkpoint push

In [ ]:
import yaml

config_path = f"{WORKING_DIR}/configs/train.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

config['hub_model_id'] = HF_REPO_ID
config['output_dir'] = CHECKPOINT_DIR  # save to Drive instead of /content/

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"hub_model_id  : {HF_REPO_ID}")
print(f"output_dir    : {CHECKPOINT_DIR}")

## 4. Train

If this cell is re-run after a session restart, it will **automatically resume** from the latest local checkpoint (if any), or pull from Hub if disk was wiped.

In [ ]:
import glob
from huggingface_hub import snapshot_download

output_dir = f"{WORKING_DIR}/outputs/checkpoint"
os.makedirs(output_dir, exist_ok=True)

has_local_checkpoint = bool(glob.glob(os.path.join(output_dir, "checkpoint-*")))

if not has_local_checkpoint:
    try:
        print(f"No local checkpoint. Trying to restore from Hub: {HF_REPO_ID}")
        snapshot_download(
            repo_id=HF_REPO_ID,
            local_dir=output_dir,
            token=os.environ["HF_TOKEN"],
        )
        print("Restored from Hub.")
    except Exception as e:
        print(f"Hub restore skipped ({e}) — starting fresh.")
else:
    latest = sorted(glob.glob(os.path.join(output_dir, "checkpoint-*")))[-1]
    print(f"Local checkpoint found: {latest}")

In [ ]:
os.chdir(f"{WORKING_DIR}/scripts")

train_result = subprocess.run([sys.executable, "train.py"], env=os.environ)
if train_result.returncode != 0:
    raise RuntimeError("Training failed — check logs above.")
print("Training complete.")

## 5. Quick sanity check — predict one sample

In [ ]:
import sys
sys.path.insert(0, f"{WORKING_DIR}/scripts")

from inference import IntentClassification

clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode="finetuned")
test_input = "I lost my credit card, how do I order a replacement?"
result = clf.predict(test_input)
print(result)

## 6. Evaluate — quick test (20 samples)

Stratified sample across intents to verify the pipeline before running the full evaluation.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score
from tqdm import tqdm

test_df = pd.read_csv(f"{WORKING_DIR}/sample_data/test.csv")
sample_df = (
    test_df.groupby("intent_name", group_keys=False)
    .apply(lambda g: g.sample(min(1, len(g)), random_state=42))
    .sample(20, random_state=42)
    .reset_index(drop=True)
)
print(f"Sampled {len(sample_df)} rows across {sample_df['intent_name'].nunique()} intents")

def quick_eval(clf, df, label, batch_size=8):
    texts = df["text"].tolist()
    y_true = df["intent_name"].tolist()
    y_pred = []
    for start in tqdm(range(0, len(texts), batch_size), desc=label):
        results = clf.predict_batch(texts[start:start + batch_size])
        for result, true_label in zip(results, y_true[start:start + batch_size]):
            y_pred.append(result["label"])
            print(f"  [{'OK' if result['label'] == true_label else 'MISS'}] "
                  f"true={true_label}  pred={result['label']}")
    acc = accuracy_score(y_true, y_pred)
    print(f"\n>>> {label} accuracy: {acc:.2f} ({int(acc*len(df))}/{len(df)})\n")
    return acc

results = {}
for mode in ("zero_shot", "finetuned"):
    clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode=mode)
    results[mode] = quick_eval(clf, sample_df, mode)

print("=== SUMMARY ===")
for mode, acc in results.items():
    print(f"  {mode:<12} {acc:.2f}")

## 7. Evaluate on full test set

Run all 3 modes and print accuracy comparison table.

In [ ]:
os.chdir(f"{WORKING_DIR}/scripts")

eval_result = subprocess.run(
    [sys.executable, "evaluate.py", "--mode", "all", "--few_shot_k", "5"],
    env=os.environ,
)
if eval_result.returncode != 0:
    raise RuntimeError("Evaluation failed — check logs above.")